# Combine exiD per-recording summary CSV files

This notebook combines all available files named like `exid_metrics_summary_recording_00.csv` through `exid_metrics_summary_recording_92.csv` into one final CSV file. Missing recording IDs are allowed.

In [ ]:
from pathlib import Path
import re
import pandas as pd


## 1. Set the input and output paths

Start Jupyter from the repository root. The summary folder is available through the relative path `results/summary`.

In [ ]:
# Paths are resolved from the repository root.
summary_dir = Path("results/summary")

output_path = summary_dir / "exid_metrics_summary_all_recordings.csv"

print("Current working directory:", Path.cwd())
print("Summary folder:", summary_dir.resolve())
print("Folder exists:", summary_dir.exists())
print("Output file:", output_path.resolve())


## 2. Find all matching summary files

The pattern also accepts duplicate Windows filenames such as `exid_metrics_summary_recording_02(1).csv`.

In [ ]:
file_pattern = re.compile(
    r"^exid_metrics_summary_recording_(\d{2})(?:\(\d+\))?\.csv$",
    re.IGNORECASE,
)

files_by_recording = {}

for path in summary_dir.iterdir():
    if not path.is_file():
        continue

    match = file_pattern.match(path.name)
    if match is None:
        continue

    recording_id = int(match.group(1))

    if 0 <= recording_id <= 92:
        files_by_recording.setdefault(recording_id, []).append(path)

print(f"Found files for {len(files_by_recording)} recording IDs.")
print("Available recording IDs:")
print(sorted(files_by_recording))


## 3. Select one file per recording

When multiple copies exist, the notebook prefers the canonical filename without `(1)`, `(2)`, and so on. Otherwise, it uses the most recently modified copy.

In [ ]:
selected_files = {}

for recording_id, candidates in files_by_recording.items():
    canonical_name = f"exid_metrics_summary_recording_{recording_id:02d}.csv"
    canonical_candidates = [
        path for path in candidates
        if path.name.lower() == canonical_name.lower()
    ]

    if canonical_candidates:
        selected = canonical_candidates[0]
    else:
        selected = max(candidates, key=lambda path: path.stat().st_mtime)

    selected_files[recording_id] = selected

    if len(candidates) > 1:
        print(
            f"Recording {recording_id:02d}: multiple files found. "
            f"Using {selected.name}"
        )

print(f"Selected {len(selected_files)} files.")


## 4. Read, validate, and combine the files

In [ ]:
summary_tables = []
reference_columns = None
loaded_ids = []

for recording_id in sorted(selected_files):
    path = selected_files[recording_id]

    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(f"Skipping empty file: {path.name}")
        continue

    if df.empty:
        print(f"Skipping file with no rows: {path.name}")
        continue

    # Check that all files have the same columns.
    if reference_columns is None:
        reference_columns = df.columns.tolist()
    elif df.columns.tolist() != reference_columns:
        missing_columns = sorted(set(reference_columns) - set(df.columns))
        extra_columns = sorted(set(df.columns) - set(reference_columns))
        raise ValueError(
            f"Column mismatch in {path.name}.\n"
            f"Missing columns: {missing_columns}\n"
            f"Extra columns: {extra_columns}"
        )

    if "recording_id" not in df.columns:
        raise ValueError(f"recording_id is missing from {path.name}")

    file_ids = set(
        pd.to_numeric(df["recording_id"], errors="coerce")
        .dropna()
        .astype(int)
        .unique()
    )

    if file_ids != {recording_id}:
        raise ValueError(
            f"{path.name} should contain recording_id {recording_id}, "
            f"but contains {sorted(file_ids)}"
        )

    summary_tables.append(df)
    loaded_ids.append(recording_id)
    print(f"Loaded recording {recording_id:02d}: {len(df):,} vehicles")

if not summary_tables:
    raise ValueError("No non-empty summary files were available to combine.")

combined_df = pd.concat(summary_tables, ignore_index=True)

sort_columns = [
    column
    for column in ["recording_id", "ego_track_id"]
    if column in combined_df.columns
]

if sort_columns:
    combined_df = combined_df.sort_values(
        sort_columns,
        kind="stable",
    ).reset_index(drop=True)

print("\nCombined shape:", combined_df.shape)
combined_df.head()


## 5. Check duplicates and missing recordings

In [ ]:
key_columns = [
    column
    for column in ["recording_id", "ego_track_id"]
    if column in combined_df.columns
]

if len(key_columns) == 2:
    duplicated_mask = combined_df.duplicated(key_columns, keep=False)
    duplicate_count = int(duplicated_mask.sum())

    print("Duplicate vehicle-summary rows:", duplicate_count)

    if duplicate_count > 0:
        display(
            combined_df.loc[duplicated_mask, key_columns]
            .sort_values(key_columns)
            .head(20)
        )
        raise ValueError(
            "Duplicate (recording_id, ego_track_id) combinations were found."
        )

expected_ids = set(range(93))
missing_ids = sorted(expected_ids - set(loaded_ids))

print("Loaded recording IDs:", loaded_ids)
print("Missing recording IDs:", missing_ids)
print("Missing IDs are acceptable when those recordings contain no merging vehicles.")


## 6. Save the final combined CSV

In [ ]:
combined_df.to_csv(output_path, index=False)

print("Saved successfully.")
print("Final CSV:", output_path.resolve())
print("Total vehicle summaries:", f"{len(combined_df):,}")
print("Number of columns:", len(combined_df.columns))
